In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve
)
from sklearn.model_selection import train_test_split

# Cargar datos
df = pd.read_csv("../data/raw/breast_cancer.csv")
X = df.drop('target', axis=1)
y = df['target']

In [2]:
# Cargar scaler y pca
with open("models/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("models/pca.pkl", "rb") as f:
    pca = pickle.load(f)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Todo cargado correctamente")
print(f"Train: {X_train_scaled.shape} | Test: {X_test_scaled.shape}")

Todo cargado correctamente
Train: (455, 30) | Test: (114, 30)


In [3]:
lr = LogisticRegression(random_state=42, max_iter=10000)
lr.fit(X_train_scaled, y_train)

# Cross-validation
cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='roc_auc')
print(f"Logistic Regression — AUC-ROC CV: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Evaluación en test
y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

print(f"\nAUC-ROC Test: {roc_auc_score(y_test, y_prob):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Maligno', 'Benigno'])}")

Logistic Regression — AUC-ROC CV: 0.9945 ± 0.0085

AUC-ROC Test: 0.9964

              precision    recall  f1-score   support

     Maligno       0.98      0.98      0.98        42
     Benigno       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



## Baseline — Logistic Regression

AUC-ROC CV: 0.9945 ± 0.0085  
AUC-ROC Test: 0.9964  
Accuracy: 98%  
Recall Maligno: 0.98 (41/42 tumores malignos correctamente identificados)

El baseline es excepcionalmente alto, consistente con la separación visual 
observada en el EDA. Los modelos subsecuentes tendrán poco margen de mejora 
en métricas absolutas, pero se evaluarán en búsqueda de mayor consistencia 
y menor varianza.

In [ ]:
modelos = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'KNN': KNeighborsClassifier()
}

resultados = {'Logistic Regression': {
    'cv_auc': cv_scores.mean(),
    'cv_std': cv_scores.std(),
    'test_auc': roc_auc_score(y_test, y_prob)
}}

for nombre, modelo in modelos.items():
    cv = cross_val_score(modelo, X_train_scaled, y_train, cv=5, scoring='roc_auc')
    modelo.fit(X_train_scaled, y_train)
    y_p = modelo.predict_proba(X_test_scaled)[:, 1]
    test_auc = roc_auc_score(y_test, y_p)
    
    resultados[nombre] = {
        'cv_auc': cv.mean(),
        'cv_std': cv.std(),
        'test_auc': test_auc
    }
    
    print(f"{nombre}")
    print(f"CV AUC: {cv.mean():.4f} ± {cv.std():.4f}")
    print(f"Test AUC: {test_auc:.4f}\n")

Random Forest
  CV AUC:   0.9875 ± 0.0129
  Test AUC: 0.9939

SVM
  CV AUC:   0.9948 ± 0.0066
  Test AUC: 0.9964

KNN
  CV AUC:   0.9892 ± 0.0131
  Test AUC: 0.9897



In [5]:
# Celda 4 — GridSearchCV
param_grids = {
    'Logistic Regression': {
        'modelo': LogisticRegression(random_state=42, max_iter=10000),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'solver': ['lbfgs', 'liblinear']
        }
    },
    'SVM': {
        'modelo': SVC(probability=True, random_state=42),
        'params': {
            'C': [0.1, 1, 10, 100],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto']
        }
    },
    'Random Forest': {
        'modelo': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5]
        }
    }
}

In [6]:
mejores_modelos = {}

for nombre, config in param_grids.items():
    grid = GridSearchCV(
        config['modelo'],
        config['params'],
        cv=5,
        scoring='roc_auc',
        n_jobs=-1
    )
    grid.fit(X_train_scaled, y_train)
    
    y_p = grid.predict_proba(X_test_scaled)[:, 1]
    test_auc = roc_auc_score(y_test, y_p)
    
    mejores_modelos[nombre] = grid.best_estimator_
    
    print(f"{nombre}")
    print(f"  Mejores params: {grid.best_params_}")
    print(f"  CV AUC:         {grid.best_score_:.4f}")
    print(f"  Test AUC:       {test_auc:.4f}\n")

Logistic Regression
  Mejores params: {'C': 0.1, 'solver': 'liblinear'}
  CV AUC:         0.9951
  Test AUC:       0.9964

SVM
  Mejores params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
  CV AUC:         0.9948
  Test AUC:       0.9964

Random Forest
  Mejores params: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
  CV AUC:         0.9875
  Test AUC:       0.9939



In [7]:
#Guardar modelo final
modelo_final = mejores_modelos['Logistic Regression']

with open("models/final_model.pkl", "wb") as f:
    pickle.dump(modelo_final, f)

print("Modelo final guardado en models/final_model.pkl")
print(f"Modelo: Logistic Regression")
print(f"Params: C=0.1, solver=liblinear")
print(f"Test AUC: 0.9964")

Modelo final guardado en models/final_model.pkl
Modelo: Logistic Regression
Params: C=0.1, solver=liblinear
Test AUC: 0.9964


## Selección del Modelo Final

Tras comparar 4 modelos con y sin tuning de hiperparámetros, se selecciona 
**Logistic Regression** (C=0.1, solver=liblinear) como modelo final por las 
siguientes razones:

- Empata con SVM en Test AUC (0.9964) con mayor CV AUC (0.9951 vs 0.9948)
- Es interpretable — sus coeficientes tienen significado clínico directo
- Es computacionalmente eficiente para el despliegue en Streamlit
- Random Forest no mejoró con tuning, descartado por menor AUC general

El modelo identifica correctamente el 98% de los tumores malignos en el 
conjunto de test, minimizando los falsos negativos que son el error más 
crítico en contexto clínico.